In [ ]:

import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from prettytable import PrettyTable
from skimage import filters

In [ ]:





# Local helpers
from src.utils.config import Config
from src.utils.helpers import init_this_notebook, p


config = Config()
init_this_notebook(config.SEED)
p("Configuration loaded", config)

# Mapping of zip files to their extraction targets
paths = { config.PATHS_TRAIN_IMAGES_ZIP: config.PATHS_TRAIN_IMAGES,
          config.PATHS_EVAL_IMAGES_ZIP: config.PATHS_EVAL_IMAGES }

p("paths", paths)

# --- exploration parameters ---
if not hasattr(config, "JSON_PATH"):
    config.JSON_PATH = config.PATHS_DATA / "train_annotations.json"

if not hasattr(config, "PATHS_JSON_ANNOTATIONS"):
    config.PATHS_JSON_ANNOTATIONS = config.PATHS_DATA / "annotations"

if not hasattr(config, "PATHS_EXPLORATION_OUTPUT"):
    config.PATHS_EXPLORATION_OUTPUT = config.PATHS_PLOTS / "exploration"
    config.PATHS_EXPLORATION_OUTPUT.mkdir(parents = True, exist_ok = True)

if not hasattr(config, "MASK_COLORS"):
    config.MASK_COLORS = {
        "individual_tree": (0, 255, 0),
        "group_of_trees": (255, 0, 0)
    }

if not hasattr(config, "TRANSFORM_TYPES"):
    config.TRANSFORM_TYPES = ["sobel", "gaussian", "laplacian", "fourier"]




In [ ]:
# Check current device
device = "cuda" if torch.cuda.is_available() else torch.device('cpu')
pin_memory = True if device == "cuda" else False
p("Device", f"Running on {device} with pin_memory: {pin_memory} ")

In [ ]:

# Bounding box from polygon
def bbox_segmentation( segmentation ):
    """
    Convert a segmentation polygon (flat or list of lists)
    into a [x_min, y_min, x_max, y_max] bounding box.
    """
    xs, ys = [], []

    # Flatten any nested polygons
    if isinstance(segmentation[0], list):
        # xs, ys = [], []
        for seg in segmentation:
            xs.extend(seg[0::2])
            ys.extend(seg[1::2])
    else:
        xs = segmentation[0::2]
        ys = segmentation[1::2]

    if not xs or not ys:
        return [0, 0, 0, 0]

    return [min(xs), min(ys), max(xs), max(ys)]


def bbox_segmentation_individual( segmentation ):
    """
    Convert segmentation(s) to one or more [x_min, y_min, x_max, y_max] bounding boxes.
    Returns a list of boxes (even if only one).
    """
    if not segmentation:
        return []

    boxes = []
    if isinstance(segmentation[0], list):
        for seg in segmentation:
            xs = seg[0::2]
            ys = seg[1::2]
            boxes.append([min(xs), min(ys), max(xs), max(ys)])
    else:
        xs = segmentation[0::2]
        ys = segmentation[1::2]
        boxes.append([min(xs), min(ys), max(xs), max(ys)])

    return boxes


def bbox_segmentation_group( segmentation, include_group_box = True ):
    """
    Convert a segmentation polygon (or list of polygons)
    into a list of bounding boxes.
    Returns a dict with 'individual_boxes' and optional 'group_box'.
    """
    if not segmentation:
        return { "individual_boxes": [], "group_box": None }

    if isinstance(segmentation[0], list):
        # multiple polygons
        all_x, all_y = [], []
        individual_boxes = []
        for seg in segmentation:
            xs = seg[0::2]
            ys = seg[1::2]
            individual_boxes.append([min(xs), min(ys), max(xs), max(ys)])
            all_x.extend(xs)
            all_y.extend(ys)
        group_box = [min(all_x), min(all_y), max(all_x), max(all_y)] if include_group_box else None
    else:
        xs = segmentation[0::2]
        ys = segmentation[1::2]
        individual_boxes = [[min(xs), min(ys), max(xs), max(ys)]]
        group_box = individual_boxes[0] if include_group_box else None

    return { "individual_boxes": individual_boxes, "group_box": group_box }
    # if not segmentation:
    #     return []
    #
    # all_x, all_y = [], []
    # boxes = []
    #
    # if isinstance(segmentation[0], list):
    #     # multiple polygons
    #     for seg in segmentation:
    #         xs = seg[0::2]
    #         ys = seg[1::2]
    #         boxes.append({
    #                 "type": "individual",
    #                 "bbox": [min(xs), min(ys), max(xs), max(ys)]
    #         })
    #         all_x.extend(xs)
    #         all_y.extend(ys)
    #     if include_group_box:
    #         boxes.append({
    #                 "type": "group",
    #                 "bbox": [min(all_x), min(all_y), max(all_x), max(all_y)]
    #         })
    # else:
    #     # single polygon
    #     xs = segmentation[0::2]
    #     ys = segmentation[1::2]
    #     boxes.append({
    #             "type": "individual",
    #             "bbox": [min(xs), min(ys), max(xs), max(ys)]
    #     })
    #     if include_group_box:
    #         boxes.append({
    #                 "type": "group",
    #                 "bbox": [min(xs), min(ys), max(xs), max(ys)]
    #         })
    #
    # return boxes


# Load annotations (now mirrors the working single-cell behavior)
def load_annotations( json_path ):
    with open(json_path, "r") as f:
        data = json.load(f)

    all_items = []
    for item in data.get("images", []):
        img_path = Path(item["file_name"])
        #width, height = item.get("width"), item.get("height")
        annotations = item.get("annotations", [])
        bboxes = []

        for ann in annotations:
            clss = ann.get("class", "unknown")
            segmentation = ann.get("segmentation", [])
            confidence = ann.get("confidence_score", 1.0)

            # if not segmentation or len(segmentation) < 4:
            #     continue

            # Each annotation is a flat polygon → one bbox
            bbox = bbox_segmentation(segmentation)
            bboxes.append({
                "class": clss,
                "bbox": bbox,
                "confidence_score": confidence
            })
            # boxes = bbox_segmentation_individual(segmentation)
            # boxes = bbox_segmentation_group(segmentation)
            #
            # # Add individual boxes
            # for box in boxes["individual_boxes"]:
            #     bboxes.append({
            #             "class": clss,
            #             "bbox": box,
            #             "bbox_type": "individual",
            #             "confidence_score": confidence
            #     })
            # # Add group box (optional)
            # if boxes["group_box"] is not None:
            #     bboxes.append({
            #             "class": clss,
            #             "bbox": boxes["group_box"],
            #             "bbox_type": "group",
            #             "confidence_score": confidence
            #     })

        all_items.append({ "image_path": img_path, "bboxes": bboxes })
    return all_items


# Unique classes
def get_unique_classes( annotations ):
    classes = { bbox["class"] for item in annotations for bbox in item["bboxes"] }
    return sorted(classes)


# Visualization helper
def show_image_with_mask( image_path, mask_path, bboxes = None, alpha = 0.3, plot_bboxes = True ):
    image = cv2.imread(str(image_path))
    if image is None:
        p("Failed to read image", image_path)
        return
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        p("Failed to read mask", mask_path)
        return

    mask_rgb = np.zeros_like(image)
    mask_rgb[:, :, 0] = mask
    overlay = cv2.addWeighted(image, 1 - alpha, mask_rgb, alpha, 0)

    if plot_bboxes and bboxes:
        for bbox in bboxes:
            # box = bbox["bbox"]
            # if isinstance(box[0], list):  # nested
            #     box = box[0]
            box = bbox.get("bbox")
            if not isinstance(box, (list, tuple)) or len(box) != 4:
                continue  # skip bad data
            x1, y1, x2, y2 = map(int, box)
            #x1, y1, x2, y2 = map(int, bbox["bbox"])

            if bbox["class"] == "individual_tree":
                color = (0, 255, 0)  # green
            elif bbox["class"] == "group_of_trees":
                #color = (255, 255, 0)  # yellow
                color = (255, 255, 0)  #if bbox.get("bbox_type") == "individual" else (255, 128, 0)
            else:
                color = (255, 0, 0)  # blue (fallback)
            #
            # color = (0, 255, 0) if bbox["class"] == "individual_tree" \
            #     else (0, 0, 255) if bbox["class"] == "group_of_trees" \
            #     else (255, 0, 0)
            #

            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 1)
            #label = bbox["class"]
            label = str(1 if bbox["class"] == "individual_tree" else 2 if bbox["class"] == "group_of_trees" else 0)
            #label = f"{bbox['class'][0].upper()}-{bbox.get('bbox_type', '')[:1].upper()}"

            org = (x1, max(10, y1 - 5))
            fontFace = cv2.FONT_HERSHEY_SIMPLEX
            fontScale = 0.4
            thickness = 1

            cv2.putText(
                    overlay,  # image to draw on
                    label,  # text string
                    org,  # (x, y) coordinates for bottom-left corner
                    fontFace,  # cv2.FONT_HERSHEY_SIMPLEX
                    fontScale,  # text size multiplier
                    color,  # (B, G, R)
                    thickness,  # line thickness
                    #lineType=cv2.LINE_AA    # (optional) anti-aliasing
            )

    plt.figure(figsize = (4, 4), dpi = 300)
    # plt.subplot(1, 2, 1)
    # plt.imshow(image)
    # plt.title("Image")
    # plt.axis("off")

    #plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title("With Mask + BBoxes")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def find_images_with_both_classes( annotations, class_a = "individual_tree", class_b = "group_of_trees" ):
    """
    Return a list of image paths that contain both class_a and class_b annotations.
    """
    results = []

    for item in annotations:
        classes_in_image = { bbox["class"] for bbox in item["bboxes"] }
        if class_a in classes_in_image and class_b in classes_in_image:
            results.append(item["image_path"])

    return results


def show_annotations_for_image( image_name, annotations, config, alpha = 0.4 ):
    """
    Display bounding boxes and mask for a specific image.
    image_name: e.g. '30.tif'
    """

    # find the image entry
    item = next((a for a in annotations if a["image_path"].name == image_name), None)
    if item is None:
        print(f"Image '{image_name}' not found in annotations.")
        return

    img_name = item["image_path"].name
    bboxes = item["bboxes"]

    print(f"\nImage: {img_name}")
    print(f"Total bounding boxes: {len(bboxes)}")

    # summarize class counts
    class_counts = { }
    for b in bboxes:
        class_counts[b["class"]] = class_counts.get(b["class"], 0) + 1

    p("Class counts", class_counts)

    # create a table of all bounding boxes
    table = PrettyTable()
    table.field_names = ["#", "Class", "BBox", "Confidence"]
    for i_bbox, bbox in enumerate(bboxes):
        table.add_row([
            i_bbox + 1,
            bbox["class"],
            bbox["bbox"],
            bbox["confidence_score"]
        ])
    print(table)

    # locate mask and image
    sample_img = config.PATHS_TRAIN_IMAGES / img_name
    sample_mask = config.PATHS_TRAIN_MASKS / img_name

    # visualize with overlay
    show_image_with_mask(sample_img, sample_mask, bboxes = bboxes, alpha = alpha, plot_bboxes = True)






In [ ]:



all_annotations = load_annotations(config.JSON_PATH)
unique_classes = get_unique_classes(all_annotations)

images_with_both = find_images_with_both_classes(all_annotations)
for img_path in images_with_both[:1]:
    p("(sample) ⚠ Mixed-class image", img_path.name, color = "yellow")

p("Unique classes", unique_classes)
images = sorted(config.PATHS_TRAIN_IMAGES.glob("*"))

target_image = "60cm_train_124.tif"

show_annotations_for_image(target_image, all_annotations, config)

# for clss in unique_classes:
#     p(f"Class: {clss}", color = "red")
#
#     # Sort by how many boxes of this class exist per image
#     sorted_annotations = sorted(
#         enumerate(annotations),
#         key = lambda x: len([b for b in x[1]["bboxes"] if b["class"] == clss]),
#         reverse = True
#     )
#
#     # Pick the image with the most of this class
#     for idx, item in sorted_annotations:
#         bboxes = [b for b in item["bboxes"] if b["class"] == clss]
#         if not bboxes:
#             continue
#
#         img_name = item["image_path"].name
#         p(img_name, f"{len(bboxes)} bounding boxes")
#
#         # Call the refactored function here
#         show_annotations_for_image(img_name, annotations, config, alpha = 0.4)
#         break


# #
# # for clss in unique_classes:
# #     p(f"Class: {clss}")
# #
# #     # Sort by how many boxes of this class exist per image
# #     sorted_annotations = sorted(
# #         enumerate(annotations),
# #         key = lambda x: len([b for b in x[1]["bboxes"] if b["class"] == clss]),
# #         reverse = True
# #     )
# #
# #     # Pick the image with the most of this class
# #     for idx, item in sorted_annotations:
# #         bboxes = [b for b in item["bboxes"] if b["class"] == clss]
# #         if not bboxes:
# #             continue
# #
# #         img_name = item["image_path"].name
# #         p(img_name, f"{len(bboxes)} bounding boxes")
# #
# #         table = PrettyTable()
# #         table.field_names = ["#", "Class", "BBox", "Confidence"]
# #         for i_bbox, bbox in enumerate(bboxes[:5]):  # limit to first few for display
# #             table.add_row([i_bbox + 1, bbox["class"], bbox["bbox"], bbox["confidence_score"]])
# #         p(table)
# #
# #         sample_img = config.PATHS_TRAIN_IMAGES / img_name
# #         sample_mask = config.PATHS_TRAIN_MASKS / img_name
# #         show_image_with_mask(sample_img, sample_mask, bboxes = bboxes, alpha = 0.4, plot_bboxes = True)
# #         break

In [ ]:


# 1. Read JSON
with open(config.JSON_PATH, "r") as f:
    data = json.load(f)

# 2. Find the target image entry
image_entry = next((img for img in data["images"] if Path(img["file_name"]).name == target_image), None)
if image_entry is None:
    raise ValueError(f"{target_image} not found in JSON.")

width, height = image_entry["width"], image_entry["height"]

# 3. Create mask (same logic as your create_masks_from_custom_json)
mask = np.zeros((height, width), dtype = np.uint8)
for ann in image_entry["annotations"]:
    segmentation = ann.get("segmentation", [])
    if not segmentation or len(segmentation) < 4:
        continue
    poly = np.array(segmentation).reshape(-1, 2).astype(np.int32)
    cv2.fillPoly(mask, [poly], color = 1)

# 4. Compute bounding boxes (one per annotation)
bboxes = []
for ann in image_entry["annotations"]:
    segmentation = ann.get("segmentation", [])
    if not segmentation or len(segmentation) < 4:
        continue
    xs = segmentation[0::2]
    ys = segmentation[1::2]
    bbox = [min(xs), min(ys), max(xs), max(ys)]
    bboxes.append(bbox)

# 5. Read image
image_path = config.PATHS_TRAIN_IMAGES / target_image
image = cv2.imread(str(image_path))
if image is None:
    raise FileNotFoundError(f"Could not read {image_path}")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Prepare mask RGB
mask_rgb = np.zeros_like(image)
mask_rgb[:, :, 0] = mask * 255
overlay = cv2.addWeighted(image, 0.7, mask_rgb, 0.3, 0)

# Draw bounding boxes
for (x1, y1, x2, y2) in bboxes:
    cv2.rectangle(overlay, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 1)

# 6. Plot
plt.figure(figsize = (15, 5), dpi = 200)
plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title("Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask, cmap = "gray")
plt.title("Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title("Image + Mask + Boxes")
plt.axis("off")

plt.tight_layout()
plt.show()



# Tree Top Detection Project Specification

## Project Overview

Develop a modular computer vision pipeline to detect and analyze tree tops in aerial or satellite imagery using the `individual_tree` and `group_of_trees` classes. The system must support independently callable methods and extensive visual exploration, focusing on experimentation and CPU efficiency.

---

## Objectives

- Detect individual trees and groups of trees with object detection techniques.
- Modularize functionalities for isolated testing, visual analysis, and reuse.
- Enable comprehensive experimentation with transformations and kernels.

---

## Data Preparation

- **Image Handling:**
  - Load individual images.
  - Utilize existing functions for reading.

- **Annotation Processing:**
  - Read JSON annotations (segmentations and bounding boxes).

- **Mask Generation:**
  - Save colored masks for each class (`individual_tree`, `group_of_trees`).
  - Save individual masks only.
  - Save group masks only.

---

## Model Training and Evaluation

- **Training Steps:**
  - Train a dedicated model on `individual_tree` masks.
  - Train a separate model on `group_of_trees` masks.
  - Experiment with ensembles and hybrid approaches.
  - Within groups, search for and label individual trees (`new_individual_tree`).

- **Evaluation:**
  - Visualize predictions and mask overlays.
  - Save all outputs for further analysis.
  - Compare methods with accuracy metrics (e.g., IoU).

---

## Prediction Pipeline

- Predict **group masks** in each image.
- Within detected groups, predict **individual tree** locations.

---

## Exploratory Methods

- **Kernels & Filters:**
  - Extract, apply, and visualize linear, binomial, and gradient kernels.

- **Transformations:**
  - Perform and visualize Fourier transformations on sample images.

- **Spatial Sampling:**
  - Apply and visualize effects of various spatial sampling strategies.

- **Modular Practices:**
  - Ensure all methods can be called independently on static samples.

---

## Computational Requirements

- Optimize all code for typical CPU devices.
- Consider efficient implementation for all steps.

---

## Outputs & Visualization

- Save masks, model results, and all transformations.
- Visualize all steps: show before/after, overlay, and highlight differences.
- Summarize findings in tables and charts.

---

## Documentation & Code Style

- Every function should have clear docstrings, inputs/outputs, and usage notes.
- Final documentation should favor Markdown and visual examples.
- Scripts and notebooks must be modular and logically organized.

---

## Example Output Comparison Table

| Task                        | Model Used | Output Saved | Visualization Provided |
|-----------------------------|------------|--------------|-----------------------|
| Individual tree detection   | Model A    | Yes          | Yes                   |
| Group of trees segmentation | Model B    | Yes          | Yes                   |
| Kernel effects              | N/A        | Yes          | Yes                   |
| Fourier transform           | N/A        | Yes          | Yes                   |

---


 # Data Handling and Visualization

In [ ]:



def load_image_and_json( image_path, all_annotations ):
    """
    Load an image and extract its true segmentation polygons and bounding boxes
    from a JSON dataset structured as:
        {
          "images": [
            {
              "file_name": "img.tif",
              "width": int,
              "height": int,
              "annotations": [
                {"class": str, "segmentation": [...], "confidence_score": float}
              ]
            }
          ]
        }

    Returns:
        image (np.ndarray): RGB image
        annotations_for_image (dict): {
            "image_path": Path,
            "annotations": [
                {
                    "class": str,
                    "bbox": [x1, y1, x2, y2],
                    "segmentation": [[x1, y1, x2, y2, ...]],
                    "confidence_score": float
                }
            ]
        }
    """
    image_path = Path(image_path)
    image_name = image_path.name

    # Read image
    image = cv2.imread(str(image_path))
    if image is None:
        p("[Error]" f"Could not read image {image_path}", color = "red")
        return None, None

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Find corresponding entry
    image_entry = next((item for item in all_annotations if Path(item["image_path"]).name == image_name), None)

    if image_entry is None:
        # raise ValueError(f"{image_name} not found in JSON.")
        p("[Warning]", f"No annotations found for {image_name}", color = "salmon")
        return None, None

    width, height = image_entry.get("width"), image_entry.get("height")

    annotations_list = []
    for ann in image_entry.get("bboxes", []):
        bbox = ann.get("bbox", [])
        segmentation = ann.get("segmentation", [])

        annotations_list.append({
            "class": ann.get("class", "unknown"),
            "bbox": bbox,
            "segmentation": segmentation,
            "confidence_score": ann.get("confidence_score", 1.0)
        })

    if not annotations_list:
        p("[Warning]", f"No valid polygons found for {image_name}", color = "salmon")

    annotations = {
        "image_path": image_name,
        "annotations": annotations_list
    }

    p("[Loaded]:", f"{image_name}: {len(annotations_list)} annotations", color = "blue")

    return image, annotations


def show_side_by_side( img1, img2, title1 = "Original", title2 = "Processed" ):
    """Display two images side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize = (10, 5))
    axes[0].imshow(img1)
    axes[0].set_title(title1)
    axes[1].imshow(img2)
    axes[1].set_title(title2)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def annotation_to_mask( annotations, image_shape ):
    """
    Convert annotation data into a binary mask and list of bounding boxes.

    Args:
        image: target image.
        annotations (dict): Contains annotation entries under "annotations" key.

    Returns:
        tuple: (mask, bboxes)
            - mask (np.ndarray): Binary mask of the same height and width as the image.
            - bboxes (list): List of bounding boxes [x1, y1, x2, y2].
    """
    mask = np.zeros(image_shape[:2], dtype = np.uint8)
    bboxes = []

    for ann in annotations.get("annotations", []):
        segmentation = ann.get("segmentation", [])
        if not segmentation or len(segmentation) < 4:
            continue

        if isinstance(segmentation[0], list):
            segmentation = segmentation[0]

        # Fill polygon mask
        poly = np.array(segmentation).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(mask, [poly], color = 1)

        # Compute bounding box
        xs = segmentation[0::2]
        ys = segmentation[1::2]
        bbox = [min(xs), min(ys), max(xs), max(ys)]
        bboxes.append(bbox)

    return mask, bboxes


def overlay_masks( image, mask, alpha = 0.4 ):
    """Overlay a mask on the original image."""
    overlay = cv2.addWeighted(image, 1 - alpha, mask, alpha, 0)
    plt.imshow(overlay)
    plt.axis("off")
    plt.show()

    # """
    # Overlay segmentation masks and bounding boxes on an image.
    #
    # Args:
    #     image (np.ndarray): RGB image array.
    #     annotations (dict): Annotation data containing 'annotations' and 'bboxes' keys.
    #     alpha (float): Transparency factor for overlay blending.
    #
    # Returns:
    #     np.ndarray: Image with overlays applied.
    # """
    #
    # if image.dtype != np.uint8:
    #     image = (image * 255).astype(np.uint8)
    #
    # # Create mask and extract bounding boxes
    # mask, bboxes = annotation_to_mask(annotations, image.shape)
    #
    # # Convert mask to RGB for blending
    # mask_rgb = np.zeros_like(image)
    # mask_rgb[:, :, 0] = mask * 255  # red channel mask
    #
    # # Blend mask with image
    # overlay = cv2.addWeighted(image, 1 - alpha, mask_rgb, alpha, 0)
    #
    # # Draw bounding boxes
    # for (x1, y1, x2, y2) in bboxes:
    #     cv2.rectangle(overlay, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 1)
    #
    # # Display overlay
    # plt.figure(figsize = (4, 4), dpi = 200)
    # plt.imshow(overlay)
    # plt.axis("off")
    # plt.show()
    #
    # return overlay


# Mask & Segmentation Exploration

In [ ]:
def generate_colored_mask( annotations, image_shape, class_colors = None ):
    """
    Create a colored segmentation mask from annotation data.
    Each class gets a unique color.
    """
    if class_colors is None:
        class_colors = config.MASK_COLORS

    mask = np.zeros((*image_shape[:2], 3), dtype = np.uint8)

    for ann in annotations.get("annotations", []):
        cls = ann.get("class", "unknown")
        segmentation = ann.get("segmentation", [])
        if not segmentation or len(segmentation) < 4:
            continue

        # Convert flat list [x1, y1, x2, y2, ...] → Nx2 polygon
        poly = np.array(segmentation, dtype = np.int32).reshape(-1, 2)

        cv2.fillPoly(mask, [poly], class_colors.get(cls, (255, 255, 255)))

    return mask


def save_class_specific_masks( annotations, image_shape, output_dir = None ):
    """
    Save masks for each class and a combined version.

    annotations:
      - JSON string with {"bboxes": [...]}
      - or dict with "bboxes" or "annotations"
      - or plain list of annotation objects [{"bbox": [...], "class": "...", ...}, ...]

    image_shape: (H, W, 3) or (H, W)
    output_dir: optional path to write PNGs
    """
    # if output_dir is None:
    #     output_dir = config.PATHS_EXPLORATION_OUTPUT
    # output_dir = Path(output_dir)
    # output_dir.mkdir(parents=True, exist_ok=True)
    #
    # mask_all = generate_colored_mask(annotations, image_shape)
    # cv2.imwrite(str(output_dir / "mask_all.png"), cv2.cvtColor(mask_all, cv2.COLOR_RGB2BGR))
    #
    # for cls in config.MASK_COLORS.keys():
    #     filtered = {"annotations": [a for a in annotations["annotations"] if a["class"] == cls]}
    #     mask = generate_colored_mask(filtered, image_shape)
    #     out_path = output_dir / f"{cls}_mask.png"
    #     cv2.imwrite(str(out_path), cv2.cvtColor(mask, cv2.COLOR_RGB2BGR))
    #     p("Saved", out_path.name)


    config = Config()

    # Resolve output directory
    if output_dir is None:
        # Prefer configured path if available, else create notebooks/exploration_output
        base_out = getattr(config, "PATHS_EXPLORATION_OUTPUT", None)
        if base_out is None:
            base_out = getattr(config, "PATHS_NOTEBOOKS", Path.cwd()) / "exploration_output"
        output_dir = Path(base_out)
    else:
        output_dir = Path(output_dir)
    output_dir.mkdir(parents = True, exist_ok = True)

    p("output_dir", output_dir, color = "cyan")

    # Normalize input to a Python object
    payload = annotations
    if isinstance(annotations, str):
        try:
            payload = json.loads(annotations)
        except Exception as e:
            p("Error", f"Failed to parse annotations JSON: {e}")
            return

    # Unify to list of anno dicts under key 'annotations' for downstream functions
    if isinstance(payload, dict):
        if "annotations" in payload and isinstance(payload["annotations"], list):
            anns = payload["annotations"]
        elif "bboxes" in payload and isinstance(payload["bboxes"], list):
            anns = payload["bboxes"]
        else:
            p("Warning", "Dict provided but missing 'annotations' or 'bboxes' list")
            return
    elif isinstance(payload, list):
        anns = payload
    else:
        p("Warning", "Unsupported annotations format")
        return

    if not anns:
        p("Warning", "No annotations found")
        return

    # Combined mask
    combined_input = { "annotations": anns }
    mask_all = generate_colored_mask(combined_input, image_shape)
    cv2.imwrite(str(output_dir / "mask_all.png"), cv2.cvtColor(mask_all, cv2.COLOR_RGB2BGR))
    p("Saved", "mask_all.png")

    # Determine classes to render
    configured_classes = list(getattr(config, "MASK_COLORS", { }).keys())
    present_classes = sorted({ a.get("class") for a in anns if "class" in a })
    classes_to_render = configured_classes if configured_classes else present_classes

    # Class-specific masks
    for cls in classes_to_render:
        filtered = { "annotations": [a for a in anns if a.get("class") == cls] }
        if not filtered["annotations"]:
            continue
        mask = generate_colored_mask(filtered, image_shape)
        out_path = output_dir / f"{cls}_mask.png"
        cv2.imwrite(str(out_path), cv2.cvtColor(mask, cv2.COLOR_RGB2BGR))
        p("Saved", out_path.name)


# def save_class_specific_masks( annotations, image_shape, output_dir = None ):
#     """
#     Save masks for each class separately and a combined version.
#     Accepts either:
#         - a dict with key 'annotations', or
#         - a plain list of annotation objects.
#     """
#
#     config = Config()
#
#     if output_dir is None:
#         output_dir = config.PATHS_EXPLORATION_OUTPUT
#     output_dir = Path(output_dir)
#     output_dir.mkdir(parents = True, exist_ok = True)
#
#     # Normalize input structure
#     anns = annotations.get("annotations") if isinstance(annotations, dict) else annotations
#
#     if anns is None:
#         p("Warning", "No annotations found or invalid format")
#         return
#
#     # Combined mask
#     mask_all = generate_colored_mask({ "annotations": anns }, image_shape)
#     cv2.imwrite(str(output_dir / "mask_all.png"), cv2.cvtColor(mask_all, cv2.COLOR_RGB2BGR))
#
#     # Class-specific masks
#     for cls in config.MASK_COLORS.keys():
#         filtered = { "annotations": [a for a in anns if a["class"] == cls] }
#         mask = generate_colored_mask(filtered, image_shape)
#         out_path = output_dir / f"{cls}_mask.png"
#         cv2.imwrite(str(out_path), cv2.cvtColor(mask, cv2.COLOR_RGB2BGR))
#         p("Saved", out_path.name)
#




# Transformation & Filters

In [ ]:
def apply_fourier_transform( image ):
    """
    Apply Fourier transform and visualize spectrum + reconstruction.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    magnitude = 20 * np.log(np.abs(fshift) + 1)
    reconstructed = np.fft.ifft2(np.fft.ifftshift(fshift)).real
    show_side_by_side(magnitude, reconstructed, "Magnitude Spectrum", "Reconstructed")
    return magnitude, reconstructed


def apply_spatial_filters( image, filter_type = 'sobel' ):
    """
    Apply common filters (sobel, gaussian, laplacian).
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    if filter_type == 'sobel':
        result = filters.sobel(gray)
    elif filter_type == 'gaussian':
        result = filters.gaussian(gray, sigma = 1)
    elif filter_type == 'laplacian':
        result = cv2.Laplacian(gray, cv2.CV_64F)
    else:
        raise ValueError(f"Unsupported filter: {filter_type}")
    show_side_by_side(gray, result, "Original", f"{filter_type.title()} Filter")
    return result


def downsample_image( image, factor = 2 ):
    """Downsample and visualize impact."""
    smaller = image[::factor, ::factor]
    show_side_by_side(image, smaller, "Original", f"Downsampled x{factor}")
    return smaller


def visualize_kernel_response( image, kernel ):
    """
    Convolve image with custom kernel and display response.
    """
    pass


# Training & Model Comparison

In [ ]:
class SimpleCNN(nn.Module):
    """Lightweight CNN for CPU segmentation experiments."""

    def __init__( self, in_channels = 3, out_channels = 1 ):
        super().__init__()
        self.encoder = nn.Sequential(
                nn.Conv2d(in_channels, 16, 3, padding = 1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, 3, padding = 1),
                nn.ReLU(),
                nn.MaxPool2d(2)
        )
        self.decoder = nn.Sequential(
                nn.ConvTranspose2d(32, 16, 2, stride = 2),
                nn.ReLU(),
                nn.ConvTranspose2d(16, out_channels, 2, stride = 2),
                nn.Sigmoid()
        )

    def forward( self, x ):
        x = self.encoder(x)
        return self.decoder(x)


def train_individual_tree_detector( train_loader, epochs = None, lr = None ):
    """
    Train simple segmentation model on CPU using individual tree masks.
    """
    epochs = epochs or config.EPOCHS
    lr = lr or config.LEARNING_RATE

    model = SimpleCNN()
    optimizer = optim.Adam(model.parameters(), lr = lr)
    loss_fn = nn.BCELoss()

    for epoch in range(epochs):
        for images, masks in train_loader:
            preds = model(images)
            loss = loss_fn(preds, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        p("Epoch", f"{epoch + 1}/{epochs} - Loss: {loss.item():.4f}")
    return model


def extract_individuals_from_groups( group_data, segmentation_data ):
    """
    Given group annotations, extract estimated individual tree masks.

    Returns:
        list of new individual mask polygons
    """
    pass


def group_then_refine_prediction( image, group_model, individual_model ):
    """
    Predict groups of trees first, then refine to individual detections.
    Visualize intermediate predictions.
    """
    pass


In [ ]:
# Prediction and Refinement

In [ ]:
def predict_groups( image, model ):
    """
    Use a model to predict group-of-trees regions.
    Returns binary mask (0/1).
    """
    model.eval()
    img_t = torch.tensor(image.transpose(2, 0, 1)).unsqueeze(0).float() / 255
    with torch.no_grad():
        pred = model(img_t).squeeze().numpy()
    mask = (pred > 0.5).astype(np.uint8)
    p("Predicted group mask", mask.shape)
    return mask


def refine_to_individuals( image, group_mask, individual_model ):
    """
    Within detected group areas, predict individual trees.
    """
    contours, _ = cv2.findContours(group_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    refined_mask = np.zeros_like(group_mask)
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        crop = image[y:y + h, x:x + w]
        pred = predict_groups(crop, individual_model)
        refined_mask[y:y + h, x:x + w] = np.maximum(refined_mask[y:y + h, x:x + w], pred)
    p("Refined individual predictions", refined_mask.shape)
    overlay_masks(image, np.dstack([refined_mask * 255] * 3))
    return refined_mask


# Notebook-Oriented Design

In [ ]:
## TODO - MUST CHECK load_annotations again
def load_annotations2( json_path ):
    with open(json_path, "r") as f:
        data = json.load(f)

    all_items = []
    for item in data.get("images", []):
        img_path = Path(item["file_name"])
        annotations = item.get("annotations", [])
        bboxes = []

        for ann in annotations:
            clss = ann.get("class", "unknown")
            segmentation = ann.get("segmentation", [])
            confidence = ann.get("confidence_score", 1.0)

            bbox = bbox_segmentation(segmentation)

            bboxes.append({
                "class": clss,
                "bbox": bbox,
                "segmentation": segmentation,  # keep polygon!
                "confidence_score": confidence
            })

        all_items.append({
            "image_path": img_path,
            "bboxes": bboxes
        })
    return all_items


all_annotations2 = load_annotations2(config.JSON_PATH)
p("\nall_annotations2", all_annotations2[0])

In [ ]:
test_image = next(config.PATHS_TRAIN_IMAGES.glob("*.tif"))

## target_image
test_image = next(
        (p for p in config.PATHS_TRAIN_IMAGES.glob("*.tif") if p.name == target_image),
        None
)

p("\ntest_image", test_image)
image, annotations = load_image_and_json(test_image, all_annotations2)
p("\nannotations", annotations)


In [ ]:

mask = generate_colored_mask(annotations, image.shape)
overlay_masks(image, mask)

save_class_specific_masks(annotations, image.shape)
# # apply_fourier_transform(image)
# # apply_spatial_filters(image, 'sobel')
# # downsample_image(image, 4)
